# optimizer-loop-on-tensor — ex1: manual SGD step: for p in self.params: p -= lr * p.grad

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-loop-on-tensor`. Running the final beacon cell reports progress against the `Optimizer: optimizer.step loop over params` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: optimizer.step loop over params` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-loop-on-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-loop-on-tensor"
DD_SUBTOPIC = "Optimizer: optimizer.step loop over params"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `for p in self.params: p.data -= lr * p.grad` — quick refresher

The bare-minimum hand-rolled SGD step is one explicit Python loop over the parameter list:

```
class SGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad         # in-place under inference_mode
```

**Why the loop is explicit.** Each parameter is a DIFFERENT tensor with a DIFFERENT shape. You cannot vectorize the update across parameters without flattening into a single contiguous buffer (which PyTorch's `_foreach_*` ops do — see `torch.optim.SGD` source for the optimized path). For a hand-rolled optimizer the explicit loop is the right form.

**Why guard `if p.grad is not None`.** `zero_grad(set_to_none=True)` (the default since PyTorch 1.11) clears grads to `None`, not to zero. A parameter that was never used in the forward pass (frozen head, masked branch) has `p.grad is None` after `zero_grad`. Skipping it is correct; trying `p.grad * self.lr` would crash.

**Why `p -= ...` and not `p.data -= ...`.** Under `@t.inference_mode()` the bare in-place op on a leaf is legal. Without the decorator you reach for `p.data -= ...` as the escape hatch — but ARENA prefers the decorator. Both produce identical numerical results.

### Exercise 1 — manual SGD step: for p in self.params: p -= lr * p.grad

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the explicit `for p in self.params: p -= self.lr * p.grad` loop inside a hand-rolled SGD optimizer's `step` method, with the `None`-grad guard for frozen / unused parameters.
> Keywords: sgd, manual-step, per-param-loop, inference-mode
> ```

**KCs targeted:** `optimizer-step-explicit-for-loop-over-params`, `guard-none-grad-from-frozen-params`

Implement `Ex1ManualSGD`. A hand-rolled SGD with an explicit per-parameter loop.

1. `__init__(self, params, lr)`: materialize `self.params = list(params)`, store `self.lr = lr`.
2. `step(self)`: decorated with `@t.inference_mode()`. For each `p in self.params`:
   - If `p.grad is None`, SKIP it (frozen or unused param).
   - Else, update IN PLACE: `p -= self.lr * p.grad`.
3. `zero_grad(self)`: set every `p.grad = None`.

Inputs/outputs at the optimizer level match torch.optim conventions — no explicit return.

The test passes a mix of TRAINABLE (`requires_grad=True`, has grad) and FROZEN (`requires_grad=False`, no grad) parameters to verify the `None`-grad guard. It also checks that the per-param loop visits EVERY param (not just the first), by giving each param a distinct gradient and verifying each moved.

In [ ]:
class Ex1ManualSGD:
    """Hand-rolled SGD with explicit per-param loop."""

    def __init__(self, params, lr: float):
        raise NotImplementedError()

    def step(self):
        raise NotImplementedError()

    def zero_grad(self):
        raise NotImplementedError()


def _test_ex1():
    # === Basic case: two trainable params, distinct gradients ===
    p_a = t.tensor([10.0, 20.0, 30.0], requires_grad=True)
    p_b = t.tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)

    opt = Ex1ManualSGD([p_a, p_b], lr=0.1)
    assert isinstance(opt.params, list), 'params must be materialized to a list (handle generator inputs)'
    assert len(opt.params) == 2

    # Manually set grads (so we don't depend on a backward pass).
    p_a.grad = t.tensor([1.0, 2.0, 3.0])
    p_b.grad = t.tensor([[0.5, 0.5], [0.5, 0.5]])

    before_a = p_a.detach().clone()
    before_b = p_b.detach().clone()

    opt.step()    # must not raise — decorator is required

    # Expected: p_a = before_a - 0.1 * grad_a; same for p_b.
    expected_a = before_a - 0.1 * t.tensor([1.0, 2.0, 3.0])
    expected_b = before_b - 0.1 * t.tensor([[0.5, 0.5], [0.5, 0.5]])
    assert t.allclose(p_a.detach(), expected_a, atol=1e-6), (
        f'p_a: expected {expected_a}, got {p_a.detach()}; '
        f'check `p -= self.lr * p.grad`'
    )
    assert t.allclose(p_b.detach(), expected_b, atol=1e-6), (
        f'p_b: expected {expected_b}, got {p_b.detach()}'
    )
    # Both params requires_grad still True.
    assert p_a.requires_grad and p_b.requires_grad

    # === None-grad guard: a frozen / unused param ===
    p_frozen = t.tensor([100.0, 100.0], requires_grad=True)
    # Don't call backward → grad is None.
    assert p_frozen.grad is None
    p_trainable = t.tensor([5.0, 5.0], requires_grad=True)
    p_trainable.grad = t.tensor([1.0, 1.0])

    opt2 = Ex1ManualSGD([p_frozen, p_trainable], lr=0.5)
    opt2.step()       # must NOT raise on the None-grad param

    # p_frozen should be UNCHANGED.
    assert t.allclose(p_frozen.detach(), t.tensor([100.0, 100.0])), (
        f'param with None grad must be skipped; was modified to {p_frozen.detach()}; '
        f'did you forget the `if p.grad is not None` guard?'
    )
    # p_trainable should have moved by lr * grad = 0.5 * 1.0 = 0.5.
    assert t.allclose(p_trainable.detach(), t.tensor([4.5, 4.5])), (
        f'trainable param should have moved by lr * grad; got {p_trainable.detach()}'
    )

    # === zero_grad sets all grads to None ===
    opt.zero_grad()
    for p in opt.params:
        assert p.grad is None, f'zero_grad must set .grad = None; got {p.grad}'

    # === Real model convergence check ===
    t.manual_seed(0)
    model = t.nn.Linear(2, 1)
    opt3 = Ex1ManualSGD(model.parameters(), lr=0.1)
    x = t.randn(32, 2)
    y = 2.0 * x[:, :1] - 3.0 * x[:, 1:2] + 1.0
    for _ in range(100):
        loss = ((model(x) - y) ** 2).mean()
        loss.backward()
        opt3.step()
        opt3.zero_grad()
    # Weight close to [2, -3], bias close to 1.
    w = model.weight.detach().flatten()
    b = model.bias.item()
    assert abs(w[0].item() - 2.0) < 0.1, f'w[0] should be ~2.0; got {w[0].item():.3f}'
    assert abs(w[1].item() + 3.0) < 0.1, f'w[1] should be ~-3.0; got {w[1].item():.3f}'
    assert abs(b - 1.0) < 0.1, f'bias should be ~1.0; got {b:.3f}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class Ex1ManualSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None
```

**Why the loop is explicit, not vectorized.** Each parameter is a tensor of a DIFFERENT shape (a `Linear(2,1)` has a `(1,2)` weight and a `(1,)` bias). You cannot subtract a single `lr * grad` block from all params at once without flattening — and flattening would lose the per-param shape needed for the next forward pass. The explicit `for p in self.params` is the canonical form.

**Why guard `if p.grad is not None`.** Two cases produce a None grad: (1) `optimizer.zero_grad(set_to_none=True)` (the PyTorch default since 1.11) clears grads to None rather than zero; (2) a parameter that was never reached by the forward pass (e.g. a frozen feature extractor, an unused embedding row) has no `.grad` populated. Hitting `None * lr` would crash. The guard skips them cleanly.

**`p -= ...` vs `p.data -= ...`.** With `@t.inference_mode()` the bare in-place update on a leaf is legal. Without the decorator you must reach for `p.data -= ...`. ARENA uses the decorator approach consistently — `torch.optim.SGD` source uses `@torch.no_grad()` which is functionally equivalent. PyTorch's optimized `_foreach_*` ops do the same thing but fused across params — they're a performance optimization of this same loop.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()